# Lend a GPU to karaokie

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karaokie-app/colab/blob/main/karaokie_worker.ipynb)


Preparing a song means separating the voice from the music and placing every
word of the lyrics against it — a few minutes of work for a graphics card.
This notebook borrows the one Colab lends you and puts it on the queue at
[karaokie.app](https://karaokie.app), then hands the finished song back.

Nothing is installed on your own machine and there is no account to make.

1. **Runtime ▸ Change runtime type ▸ T4 GPU** — it works without one, but a
   song then takes something like ten times as long.
2. Run the two cells below, in order.
3. Leave the tab open. Closing it, or Colab reclaiming the machine, stops the
   worker; a song caught half-done simply goes back on the queue for somebody
   else a few minutes later, so there is nothing to tidy up.


In [ ]:
# 1 — is this runtime any use?
#
# Two things decide that, and Colab gives out both most of the time and
# neither some of the time. Better to find out here than three minutes into
# a song.
import subprocess

def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

gpu = sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
card = gpu.stdout.strip().splitlines()[0] if gpu.returncode == 0 and gpu.stdout.strip() else ''
if card:
    print('GPU       ', card)
else:
    print('GPU        none. Runtime > Change runtime type > T4 GPU, then run this again.')
    print('           It will still work on the processor, only far more slowly.')

print('YouTube    asking...')
sh('pip -q install -U yt-dlp')
# The same request the pipeline makes: music.youtube.com, over IPv4.
probe = sh("yt-dlp -4 --skip-download --no-warnings --print '%(title)s' "
           'https://music.youtube.com/watch?v=sQtnhwU2R9Y')
if probe.returncode == 0 and probe.stdout.strip():
    print('YouTube    ok --', probe.stdout.strip().splitlines()[-1])
else:
    said = (probe.stderr or '').strip().splitlines()
    print('YouTube    would not answer this machine:')
    print('          ', said[-1] if said else 'no answer at all')
    print()
    print('           Colab addresses are sometimes asked to prove they are not a')
    print('           bot, and a worker that cannot fetch a song is no use to the')
    print('           queue. Runtime > Disconnect and delete runtime, then connect')
    print('           again: the next machine has a different address and usually')
    print('           a different answer.')


In [ ]:
# 2 — run the worker.
#
# One binary, checked against the published checksum and started. The first
# few minutes are it fetching its own Python, ffmpeg and the audio models;
# after that it takes a song, prepares it, uploads it and asks for another.
#
# Press the stop button on this cell to stop it.
import os, subprocess

card = subprocess.run('nvidia-smi --query-gpu=name --format=csv,noheader',
                      shell=True, capture_output=True, text=True).stdout.strip()
# How this machine appears in the pool at karaokie.app/worker.
os.environ['KARAOKIE_NAME'] = f'colab {card.splitlines()[0]}' if card else 'colab'

!curl -fsSL https://karaokie.app/install.sh | sh


### While it runs

Each song is logged as it goes: fetched, separated, aligned, uploaded. A run
of several hours fills this cell with a lot of text — **Runtime ▸ Clear
output** if the tab starts to feel heavy; it does not interrupt the worker.

### When it stops

Colab hands the machine back after a while — sooner if the browser tab has
been idle, and after twelve hours at the outside. Nothing needs cleaning up:
a song that was being prepared when the machine went away loses its lease and
is offered to the next worker a few minutes later.

Colab is meant for interactive work, and a runtime that spends every hour of
every day at full tilt is the sort of thing its limits exist for. Run it while
you are around, stop it when you are not.

See who else is working, and every other way to run one, at
[karaokie.app/worker](https://karaokie.app/worker).
